In [1]:
# !pip install icrawler

In [ ]:
# 이미지 다운로드 하기
from icrawler.builtin import BingImageCrawler
import os

def donwload_img(save_path, keyword, max_num):
	# 저장할 폴더 생성
	# save_path = '../../data/dataset_imagenet/tteokbokki'
	save_path = save_path
	if not os.path.exists(save_path):
			os.makedirs(save_path)

	# 크롤러 설정 (Bing 크롤러가 구글보다 제약이 적어 자주 사용됩니다)
	bing_crawler = BingImageCrawler(storage={'root_dir': save_path})

	bing_crawler.crawl(keyword=keyword, max_num=max_num)

	print(f"다운로드가 완료되었습니다. 저장 경로: {save_path}")

# donwload_img('../../data/dataset_imagenet/kimbap','김밥', 100)
# donwload_img('../../data/dataset_imagenet/tteokbokki','떡볶이', 100)

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = '../../data/dataset_imagenet'

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='training',
	seed=1000,
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

# 검증 데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='validation',
	seed=1000,
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

class_names = train_ds.class_names


Found 171 files belonging to 2 classes.
Using 137 files for training.
Found 171 files belonging to 2 classes.
Using 34 files for validation.


In [4]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(20).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [5]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False, weights='imagenet')

# 데이터 증강=>이미지를돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업 
data_augmentation = tf.keras.Sequential([
	layers.RandomFlip('horizontal_and_vertical'),
	layers.RandomRotation(0.2),
	layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
	layers.Input((224,224,3)),
	data_augmentation, 
	# 정규화를 0~1이 아닌 -1~1로 해야 함. 왜? imagenet에서 그렇게 했기 때문에 맞춰줘야함
	layers.Rescaling(1./127.5, offset=-1), 
	base_model, 
	layers.GlobalAveragePooling2D(),
	layers.Dense(128, activation='relu'),
	layers.Dropout(0.2),
	layers.Dense(2, activation='softmax')
])

In [6]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
	metrics=['accuracy'])

In [ ]:
callbacks = [
	# 조기 종료
	EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
	# 가장 좋았던 성능을 저장 => 중간에 중지해도 중지전까지 최고 성능을 가져옴
	ModelCheckpoint(filepath='전이학습_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
	# 학습률을 고정 값이 아닌 상황에 따라 줄어들도록 조절
	ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

In [8]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1, callbacks=callbacks)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6631 - loss: 0.5851
Epoch 1: val_accuracy improved from None to 0.94118, saving model to best_fs_mnist_model.keras

Epoch 1: finished saving model to best_fs_mnist_model.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.7737 - loss: 0.4293 - val_accuracy: 0.9412 - val_loss: 0.4526 - learning_rate: 0.0010
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9475 - loss: 0.1677 
Epoch 2: val_accuracy did not improve from 0.94118
5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - accuracy: 0.9197 - loss: 0.3625 - val_accuracy: 0.9118 - val_loss: 0.5754 - learning_rate: 0.0010
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9505 - loss: 0.1288
Epoch 3: val_accuracy did not improve from 0.94118
5/5 ━━━━━━━━━━━━━━━━━━━━ 12s 2s/step - accuracy: 0.9489 - loss: 0.1344 - val_accuracy: 0.9412 - val_loss: 0.8091 - learning_rate: 0.0010
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9719 - loss: 0.1081
Epoch 

In [9]:
import requests
import numpy as np
from io import BytesIO
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import mobilenet_v2

def predicts_url(url):
	# 이미지 정보를 가져옴
	response = requests.get(url)
	# 가져온 이미지를 바이트 형태로 변환
	img = image.load_img(BytesIO(response.content), target_size=(224,224))

	# 이미지 전처리 => 입력 형태 맞추기, 이미지를 배열로
	X = image.img_to_array(img)
	X = np.expand_dims(X, axis=0)

	# 예측
	predicts = model.predict(X)
	
	print('-'*30)
	for i in range(2):
		predict = predicts[0][i]
		print(f'{class_names[i]} : {predict*100:.2f}%')
	print('-'*30)

In [10]:
# 김밥
predicts_url('https://lh4.googleusercontent.com/proxy/YBicREfizdGpPHIZTWBhzPlJ46-6knDM_k2w7ZeilK4ngmnf1CiG3fc500HGTPIEkw3F9ao4lT-TFheDpiDRwBg')
predicts_url('https://image.greating.co.kr/CO/contents/202505/22/58563F38E9F248DB9AABCC37AB246BBC.jpeg')
# 떡볶이
predicts_url('https://cdn.kihoilbo.co.kr/news/photo/202008/880134_302005_3430.png')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
------------------------------
kimbap : 99.86%
tteokbokki : 0.14%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
------------------------------
kimbap : 100.00%
tteokbokki : 0.00%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
------------------------------
kimbap : 0.13%
tteokbokki : 99.87%
------------------------------
